# 05 · Model contracts on historical stock observations

**Objective:** carry the reconciled option/stock/cash accounting into two dated stock-price windows, using synthetic European model contracts and strictly earlier volatility calibration.

The underlying observations are audited FMP split-adjusted closes for JPM, PNC and GS. **The option premiums are model prices, not historical option quotes.** Historical stock paths are observed outcomes, not Q simulations. The study does not claim an executable listed-option backtest.

The fixed cases are short model calls with a five-session stock hedge, a long-equity benchmark, and the same equities with purchased model puts. Protective puts are **not automatically delta-neutralized**. The two equity cases use identical shares and initial capital.

This notebook runs independently from a fresh kernel after the audited underlying-price CSV exists. It reads cached prices only; it never reads or prints FMP credentials. The separate command-line runner writes permanent CSV/JSON/report artifacts.

## 1 · Load the accounting implementation and audited prices

The source is `data/market/prices_split_adjusted.csv`, accompanied by source-field, adjustment and hash metadata. The price table contains observed stock prices; the option positions and rules remain explicit model assumptions.

In [ ]:
from hedging_harness.notebook_views import run
_ = run('05_model_contract_replay')


## 2 · Freeze model inputs using prior data only

The predefined windows are 2019-01-02 through 2019-03-29 and 2020-02-03 through 2020-04-30. For each window, estimate fixed volatility from **252 log returns requiring 253 closing prices strictly before inception**, annualized by `sqrt(252)`.

This historical realized-volatility proxy is used as model sigma. It is not a historical implied volatility, and the constant 4% financing rate is an illustrative assumption rather than a historical rate series.

Each option strikes at its underlying's inception close and expires at the window's last declared session. Quantities correspond to $100,000 underlying notional per name. The equity comparisons share $300,000 initial capital. The short-option liability account starts at zero capital and has dollar P&L rather than a comparable equity investment return.

## 3 · Inspect the account outcomes

Stock execution is assumed at the observed close plus the stated proportional fee. Options are model-priced at inception and cash-settle at expiry. Core equity and option hedge shares are separate sleeves, with stock execution netted by underlying.

`terminal_model_pnl` includes entry premiums, interest, stock trades, stock costs and settlement. `max_drawdown_dollars` describes the single observed model-account NAV history; it is not a Monte Carlo expected shortfall estimate. `peak_borrowing` measures observed funding requirements rather than margin adequacy.

The short-call liability and the long-equity portfolio have different mandates and capital baselines. Compare the equity and protective-put cases directly; use the short-call case to inspect replication/accounting behavior.

## 4 · Reconcile representative dates

Every daily cash balance is independently reconstructed from the executed-stock trade records and instrument-level opening/settlement events. Final option payoffs are recomputed from observed terminal prices and signed quantities.

The representative rows below include inception, the first scheduled rebalance and expiry. Reconstruct each as:

**previous cash + interest + option entry + option settlement + core stock cash flow + hedge stock cash flow − stock cost = recorded cash.**

At inception, model-option cash flows are offset by signed option value in NAV. At expiry, the stock is liquidated and no option liability or asset remains. Receiving a written-option premium is not immediate profit, and a purchased put's premium is protection cost.

## 5 · Check causality, lifecycle and accounting

For each of the six runs, the audit checks cash aggregation, settlement payoff, final flat holdings and zero option value. A causality check modifies only later stock observations and confirms that earlier decisions and ledger rows remain identical. Volatility calibration uses a cutoff strictly before replay inception.

These checks establish the small **model-contract accounting replay**. They do not establish historical option quote coverage, American exercise handling, executable fills, unbiased sector membership, or generalization across many independent crises.

## 6 · Record the exact study and export when needed

The in-memory manifest contains source/data hashes, prior-data cutoffs, each option position, the equity holdings, execution assumptions and full per-run provenance. The compact display omits repeated per-run detail; it remains available in `study['manifest']['runs']`.

For permanent CSV, plot and Markdown artifacts, run `.venv/bin/python scripts/run_model_contract_replay.py`. It writes a new `results/model-contract-replay-*` directory without overwriting previous results. No credentials are stored in its output.

## Research boundary

The underlying observations are real historical stock closes, and the fixed model volatility estimates use earlier observations only. The options themselves are synthetic fixed-volatility European cash-settled contracts. Dividends, option spreads, margin, borrow fees and corporate-action event processing remain outside the account.

This is the intermediate replay explicitly allowed when historical option quotes are unavailable. It demonstrates accounting and causal mechanics on a normal and a stress window. It does not replace market-option data, broad out-of-sample validation, forecast tests or 40–50-name generalization.